## MODELOS ENTRENADOS

In [1]:
from scipy.ndimage import binary_erosion
from scipy.spatial.distance import directed_hausdorff
import numpy as np
import cv2

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

def resize_for_hausdorff(pred, gt, max_size=512):
    if max(pred.shape) > max_size:
        scale = max_size / max(pred.shape)
        new_h = int(pred.shape[0] * scale)
        new_w = int(pred.shape[1] * scale)
        pred = cv2.resize(pred.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        gt   = cv2.resize(gt.astype(np.uint8),   (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return pred, gt

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [2]:
import time
import torch

def measure_inference(predictor, image, point_coords, point_labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return masks, scores, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2

    start = time.time()
    predictor.set_image(img_path)
    predictor.model.set_classes(text=["polyp"])
    predictor.prompts["text"] = [text_prompt]
    results = predictor()
    
    latency = (time.time() - start) * 1000

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [4]:
import os
import shutil
import random

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"

def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "images")
    masks_dir = os.path.join(dataset, "instances")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "masks",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "images", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".png"), os.path.join(dataset, "masks",  split, name + ".png"))

split_dataset(dataset)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry
import cv2
import numpy as np
import os

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_file in os.listdir(self.images_dir):
            if not img_file.endswith(".jpg"):
                continue
            img_name  = os.path.splitext(img_file)[0]
            img_path  = os.path.join(self.images_dir, img_file)
            mask_path = os.path.join(self.masks_dir, img_name + ".png")
            if not os.path.exists(mask_path):
                continue
            gt = cv2.imread(mask_path, cv2.IMREAD_COLOR)
            if gt is None or gt.sum() == 0:
                continue
            self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = cv2.resize(mask, (256, 256))
        mask_binary = (mask.sum(axis=2) > 0).astype(np.float32)
        mask = torch.tensor(mask_binary).unsqueeze(0)

        gt_full = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        gt_full = cv2.cvtColor(gt_full, cv2.COLOR_BGR2RGB)
        gt_bool = gt_full.sum(axis=2) > 0
        coords  = np.argwhere(gt_bool)

        if len(coords) == 0:
            cx = self.img_size / 2
            cy = self.img_size / 2
        else:
            ymin, xmin = coords.min(axis=0)
            ymax, xmax = coords.max(axis=0)
            orig_h, orig_w = gt_full.shape[:2]
            cx = (xmin + (xmax - xmin) / 2) * (self.img_size / orig_w)
            cy = (ymin + (ymax - ymin) / 2) * (self.img_size / orig_h)

        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


def bce_dice_loss(pred, target, bce_weight=0.5):
    bce = nn.BCEWithLogitsLoss()(pred, target)
    pred_sig = torch.sigmoid(pred)
    intersection = (pred_sig * target).sum()
    dice = 1 - (2 * intersection + 1) / (pred_sig.sum() + target.sum() + 1)
    return bce_weight * bce + (1 - bce_weight) * dice


def train_sam(dataset_path, weights_path, output_weights, vit, epochs=50, batch_size=4):
    sam = sam_model_registry[vit](checkpoint=weights_path)
    sam.to(device)

    for param in sam.image_encoder.parameters():
        param.requires_grad = False
    for param in sam.prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam.mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            images, masks  = images.to(device), masks.to(device)
            points, labels = points.to(device), labels.to(device)

            loss_batch = 0
            with torch.no_grad():
                image_embeddings = sam.image_encoder(images)

            for i in range(images.shape[0]):
                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam.prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                low_res_masks, _ = sam.mask_decoder(
                    image_embeddings=image_embeddings[i].unsqueeze(0),
                    image_pe=sam.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                )

                loss_batch += bce_dice_loss(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save(sam.state_dict(), output_weights)
    return output_weights


## SAM 1 BASE ENTRENAMIENTO

In [6]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam_b_mapillary.pt"


def evaluate_model(dataset, weights_path):
    sam = sam_model_registry["vit_b"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "images", "test")
    masks_dir  = os.path.join(dataset, "masks",  "test")

    for img_file in os.listdir(images_dir):
        img_name  = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        gt_mask = cv2.cvtColor(gt_mask, cv2.COLOR_BGR2RGB)
        gt_mask = gt_mask.sum(axis = 2) > 0

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin
        central_point = np.array([[xmin + w / 2, ymin + h / 2]])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference(predictor, image, central_point, np.array([1]))

        if masks is None or len(masks) == 0:
            continue

        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_b_mapillary"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)
         
  
trained_weights = train_sam(dataset, weights, output_weights, vit="vit_b")
evaluate_model(dataset, trained_weights)


Usando GPU: NVIDIA GeForce RTX 3090


c:\users\danieltalavera\desktop\trabajo_fin_de_grado\segment-anything\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.loa

KeyboardInterrupt: 

## SAM 1 GRANDE

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam_l_mapillary.pt"


def evaluate_model(dataset, weights_path):
    sam = sam_model_registry["vit_l"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "images", "test")
    masks_dir  = os.path.join(dataset, "masks",  "test")

    for img_file in os.listdir(images_dir):
        img_name  = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        gt_mask = cv2.cvtColor(gt_mask, cv2.COLOR_BGR2RGB)
        gt_mask = gt_mask.sum(axis = 2) > 0

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin
        central_point = np.array([[xmin + w / 2, ymin + h / 2]])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference(predictor, image, central_point, np.array([1]))

        if masks is None or len(masks) == 0:
            continue

        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_l_mapillary"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)
         
  
trained_weights = train_sam(dataset, weights, output_weights, vit="vit_l")
evaluate_model(dataset, trained_weights)


Usando GPU: NVIDIA GeForce RTX 3090
Epoch 1/50 - Loss: 0.1208
Epoch 2/50 - Loss: 0.0002
Epoch 3/50 - Loss: 0.0001
Epoch 4/50 - Loss: 0.0001
Epoch 5/50 - Loss: 0.0001
Epoch 6/50 - Loss: 0.0001
Epoch 7/50 - Loss: 0.0001
Epoch 8/50 - Loss: 0.0001
Epoch 9/50 - Loss: 0.0001
Epoch 10/50 - Loss: 0.0001
Epoch 11/50 - Loss: 0.0001


KeyboardInterrupt: 